# Rank Classifier v2 — Real YOLO Crops

Train a **MobileNetV3-Small** rank classifier on corner crops extracted by the YOLO card detector from real photos.

**Why v2:** The original classifier was trained on Kaggle bounding-box crops (full cards). This version trains on actual 128×128 corner crops produced by the YOLO pipeline — the same format the Flutter app uses at inference time.

**Dataset:**  — ~5,100 crops across 13 rank classes.  
**Runtime:** GPU recommended (RunPod or Colab T4). CPU is fine but slower.

## 1. Setup

In [ ]:
# ── Install deps (RunPod / bare machine) ────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tensorflow", "matplotlib", "scikit-learn", "Pillow", "seaborn"], check=True)

import os, json, shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report

print(f"TensorFlow {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
import os
if os.path.isdir("/workspace"):
    SAVE_DIR = "/workspace/24_game_models"           # RunPod
    DATA_DIR = "/workspace/real_crops_v2"
elif os.path.isdir("/content/drive/MyDrive"):
    SAVE_DIR = "/content/drive/MyDrive/24_game_models"  # Colab
    DATA_DIR = "training/data/real_crops_v2"
else:
    SAVE_DIR = "outputs/rank_v2"                     # local fallback
    DATA_DIR = "training/data/real_crops_v2"

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoints → {SAVE_DIR}")
print(f"Dataset     → {DATA_DIR}")

IMG_SIZE   = 128
BATCH_SIZE = 32

## 2. Load Dataset

In [ ]:
# ── Discover classes ─────────────────────────────────────────────────────────
CLASS_NAMES = sorted(os.listdir(DATA_DIR))
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
assert NUM_CLASSES == 13, f"Expected 13 classes, got {NUM_CLASSES}"

# Save class names alongside model for Flutter index mapping
with open(os.path.join(SAVE_DIR, "class_names_v2.json"), "w") as f:
    json.dump(CLASS_NAMES, f)
print("class_names_v2.json saved")

In [ ]:
# ── Count crops per class to compute class weights ───────────────────────────
counts = {c: len(list(Path(DATA_DIR, c).glob("*.jpg"))) for c in CLASS_NAMES}
total  = sum(counts.values())
print("Crops per class:")
for c, n in counts.items():
    print(f"  {c:>3}: {n:4d}")
print(f"  Total: {total}")

# Class weights: inverse frequency, normalised so mean weight = 1
class_weights = {i: total / (NUM_CLASSES * counts[c]) for i, c in enumerate(CLASS_NAMES)}
print("
Class weights (highest = rarest class):")
for i, c in enumerate(CLASS_NAMES):
    print(f"  {c:>3}: {class_weights[i]:.2f}")

In [ ]:
# ── Build tf.data datasets ────────────────────────────────────────────────────
full_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    class_names=CLASS_NAMES,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    seed=42,
    shuffle=True,
)

# 80/20 split
n_batches  = len(full_ds)
n_val      = max(1, int(n_batches * 0.20))
n_train    = n_batches - n_val

train_ds = full_ds.take(n_train)
val_ds   = full_ds.skip(n_train)

# Normalise to [0, 1] and prefetch
AUTOTUNE = tf.data.AUTOTUNE
norm     = tf.keras.layers.Rescaling(1.0 / 255)
train_ds = train_ds.map(lambda x, y: (norm(x), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (norm(x), y),   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

print(f"Train batches: {n_train}  Val batches: {n_val}")

## 3. Build Model

In [ ]:
# ── Data augmentation (applied only during training) ─────────────────────────
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.06),       # ±21°
    tf.keras.layers.RandomBrightness(0.25),
    tf.keras.layers.RandomContrast(0.25),
    tf.keras.layers.RandomZoom(0.10),
], name="augmentation")

# ── MobileNetV3-Small backbone ────────────────────────────────────────────────
base = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
    include_preprocessing=False,   # we handle normalisation ourselves
)
base.trainable = False

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = data_aug(inputs)
x       = base(x, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
model   = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 4. Resume from Checkpoint

Check for existing checkpoints and skip completed training phases.

In [ ]:
import json

PHASE1_CKPT   = os.path.join(SAVE_DIR, "rank_v2_phase1.keras")
BEST_CKPT     = os.path.join(SAVE_DIR, "rank_v2_best.keras")
LAST_CKPT     = os.path.join(SAVE_DIR, "rank_v2_last.keras")
STATE_FILE    = os.path.join(SAVE_DIR, "training_state_v2.json")

skip_phase1   = False
resume_epoch  = 0

if os.path.exists(LAST_CKPT):
    print(f"Resuming from {LAST_CKPT}")
    model = tf.keras.models.load_model(LAST_CKPT)
    if os.path.exists(STATE_FILE):
        state = json.load(open(STATE_FILE))
        resume_epoch = state.get("epoch", 0)
        print(f"  Last completed epoch: {resume_epoch}")
    skip_phase1 = True
elif os.path.exists(PHASE1_CKPT):
    print(f"Phase 1 checkpoint found — loading and skipping to Phase 2")
    model = tf.keras.models.load_model(PHASE1_CKPT)
    skip_phase1 = True
else:
    print("No checkpoint found — starting from scratch")

## 5. Phase 1: Train Head (Frozen Backbone)

5 epochs, only the Dense classification head is updated.
Fast convergence, no risk of destroying ImageNet features.

In [ ]:
if skip_phase1:
    print("Skipping Phase 1 — checkpoint loaded.")
else:
    print("Phase 1: training classification head only (backbone frozen)")
    h1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=5,
        class_weight=class_weights,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(
                PHASE1_CKPT, monitor="val_accuracy",
                save_best_only=True, verbose=1),
        ],
    )
    print(f"Phase 1 best val accuracy: {max(h1.history['val_accuracy']):.4f}")

## 6. Phase 2: Fine-tune Full Model

Unfreeze all layers, lower LR to 1e-4.  
**Early stopping** (patience 10) keeps RunPod costs down.  
Best and last checkpoints saved every epoch.

In [ ]:
class SaveState(tf.keras.callbacks.Callback):
    """Write completed epoch index to disk so training can resume after disconnect."""
    def on_epoch_end(self, epoch, logs=None):
        with open(STATE_FILE, "w") as f:
            json.dump({"epoch": epoch + 1, "val_accuracy": float(logs.get("val_accuracy", 0))}, f)

# Unfreeze all layers
for layer in model.layers:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

PHASE2_EPOCHS = 60

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=10,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        BEST_CKPT, monitor="val_accuracy",
        save_best_only=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        LAST_CKPT, save_best_only=False, verbose=0),
    SaveState(),
]

print(f"Phase 2: fine-tuning all layers (max {PHASE2_EPOCHS} epochs, early stop patience=10)")
h2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    initial_epoch=resume_epoch,
    class_weight=class_weights,
    callbacks=callbacks_p2,
)
print(f"Phase 2 best val accuracy: {max(h2.history['val_accuracy']):.4f}")

## 7. Confusion Matrix

In [ ]:
y_true, y_pred = [], []
for imgs, labels in val_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

acc = np.mean(y_true == y_pred)
print(f"Validation accuracy: {acc:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Rank Classifier v2 — val accuracy {acc:.3f}")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix_v2.png"), dpi=150)
plt.show()

## 8. Export to TFLite

In [ ]:
# Load best checkpoint for export
best_model = tf.keras.models.load_model(BEST_CKPT)

# FP16 TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

TFLITE_PATH = os.path.join(SAVE_DIR, "rank_classifier_v2.tflite")
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_PATH) / (1024 * 1024)
print(f"Exported: {TFLITE_PATH}  ({size_mb:.1f} MB)")

## 9. Verify TFLite

In [ ]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
inp_det  = interpreter.get_input_details()[0]
out_det  = interpreter.get_output_details()[0]
print(f"Input:  shape={inp_det['shape']}, dtype={inp_det['dtype']}")
print(f"Output: shape={out_det['shape']}, dtype={out_det['dtype']}")

# Quick sanity check on a few val images
correct, total_checked = 0, 0
for imgs, labels in val_ds.take(3):
    for img, label in zip(imgs.numpy(), labels.numpy()):
        inp = img[np.newaxis].astype(np.float32)
        interpreter.set_tensor(inp_det["index"], inp)
        interpreter.invoke()
        pred = int(np.argmax(interpreter.get_tensor(out_det["index"])[0]))
        correct += (pred == label)
        total_checked += 1
print(f"TFLite sanity check: {correct}/{total_checked} correct")
print(f"\nDeploy to: flutter_app/assets/models/rank_classifier.tflite")
print("Class order (verify against rank_classifier.dart _indexToValue):")
for i, c in enumerate(CLASS_NAMES):
    print(f"  index {i} → {c}")